
# Preprocessing_timebased_v4.ipynb

Notebook preprocess mới, dùng **đúng các file trong thư mục `Data/`** theo yêu cầu.

## Mục tiêu
- chuẩn hóa raw sources
- giữ lại tinh thần xử lý cũ:
  - chuẩn hóa date
  - xử lý missingness
  - xử lý `bd`
  - fill categorical bằng `Unknown`
  - sửa anomaly transaction
- tách logs thành 2 nhánh:
  - historical logs
  - March logs
- xuất clean tables cho notebook Feature Engineering

## Core stance
- Core final model sẽ ưu tiên:
  - `transactions` + `transactions_v2` (combined)
  - `members_v3`
- Logs được giữ riêng để dùng cẩn trọng ở bước sau

## Train/Val/Inf split
- TRAIN_MONTHS : 2015-01 → 2016-12  (n=24)
- VAL_MONTHS   : 2017-01 → 2017-02  (n=2)
- INF_MONTH    : 2017-03  (cutoff = 2017-03-31)
- GRACE_DAYS   : 30


In [15]:

# ===== 1. Imports =====
from pathlib import Path
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)


In [16]:

# ===== 2. Paths =====
DATA_DIR = Path("Data")
DATA_DIR.mkdir(exist_ok=True, parents=True)

RAW_DIR = Path("Data")

# Both transaction files are required
TRANSACTIONS_V1_PATH    = RAW_DIR / "transactions.parquet"      # historical (2015-2016 bulk)
TRANSACTIONS_V2_PATH    = RAW_DIR / "transactions_v2.csv"   # competition update (2017)
MEMBERS_V3_PATH         = RAW_DIR / "members_v3.csv"
USER_LOGS_HIST_PATH     = RAW_DIR / "user_logs.parquet"
USER_LOGS_MARCH_PATH    = RAW_DIR / "user_logs_v2.parquet"

for p in [
    TRANSACTIONS_V1_PATH,
    TRANSACTIONS_V2_PATH,
    MEMBERS_V3_PATH,
    USER_LOGS_HIST_PATH,
    USER_LOGS_MARCH_PATH,
]:
    print(f"{p}: {'FOUND' if p.exists() else 'MISSING'}")


Data\transactions.parquet: FOUND
Data\transactions_v2.csv: FOUND
Data\members_v3.csv: FOUND
Data\user_logs.parquet: FOUND
Data\user_logs_v2.parquet: FOUND


In [17]:

# ===== 3. Load raw sources =====
for p in [TRANSACTIONS_V1_PATH, TRANSACTIONS_V2_PATH, MEMBERS_V3_PATH,
           USER_LOGS_HIST_PATH, USER_LOGS_MARCH_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

# ── date parser: both files store dates as int64 YYYYMMDD ──
def parse_yyyymmdd(df, cols):
    """Convert YYYYMMDD integer columns to datetime64."""
    for col in cols:
        if col in df.columns:
            s = df[col].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
            valid = s.str.fullmatch(r"\d{8}", na=False)
            s = s.where(valid, other=pd.NA)
            df[col] = pd.to_datetime(s, format="%Y%m%d", errors="coerce")
    return df

TX_DATE_COLS = ["transaction_date", "membership_expire_date"]

tx_v1 = parse_yyyymmdd(pd.read_parquet(TRANSACTIONS_V1_PATH), TX_DATE_COLS)
tx_v2 = parse_yyyymmdd(pd.read_csv(TRANSACTIONS_V2_PATH), TX_DATE_COLS)

# ── combine, deduplicate, sort ──
transactions_raw = (
    pd.concat([tx_v1, tx_v2], ignore_index=True)
    .drop_duplicates()
    .dropna(subset=["transaction_date", "membership_expire_date"])
    .sort_values(["msno", "transaction_date", "membership_expire_date"])
    .reset_index(drop=True)
)

members_v3      = pd.read_csv(MEMBERS_V3_PATH)
user_logs_hist  = pd.read_parquet(USER_LOGS_HIST_PATH)
user_logs_march = pd.read_parquet(USER_LOGS_MARCH_PATH)

print(f"tx_v1 (transactions)    : {tx_v1.shape}")
print(f"tx_v2 (transactions_v2) : {tx_v2.shape}")
print(f"transactions_raw (combined): {transactions_raw.shape}")
print(f"  unique users          : {transactions_raw['msno'].nunique():,}")
print(f"  transaction_date      : {transactions_raw['transaction_date'].min()} → {transactions_raw['transaction_date'].max()}")
print(f"  expire_date           : {transactions_raw['membership_expire_date'].min()} → {transactions_raw['membership_expire_date'].max()}")
print(f"members_v3              : {members_v3.shape}")
print(f"user_logs_hist          : {user_logs_hist.shape}")
print(f"user_logs_march         : {user_logs_march.shape}")

print("\nExpiry distribution (2015-01 to 2017-06):")
print(
    transactions_raw["membership_expire_date"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
    .loc["2015-01":"2017-06"]
)

display(transactions_raw.head())
display(members_v3.head())
display(user_logs_hist.head())
display(user_logs_march.head())


tx_v1 (transactions)    : (547746, 9)
tx_v2 (transactions_v2) : (1431009, 9)
transactions_raw (combined): (1978751, 9)
  unique users          : 1,315,711
  transaction_date      : 2015-01-01 00:00:00 → 2017-03-31 00:00:00
  expire_date           : 1970-01-01 00:00:00 → 2036-10-15 00:00:00
members_v3              : (6769473, 6)
user_logs_hist          : (106543, 9)
user_logs_march         : (396362, 9)

Expiry distribution (2015-01 to 2017-06):
membership_expire_date
2015-01        692
2015-02      10739
2015-03      13713
2015-04      15783
2015-05      14490
2015-06      18560
2015-07      16170
2015-08      17254
2015-09      18290
2015-10      19097
2015-11      20027
2015-12      22196
2016-01      22502
2016-02      20902
2016-03      23163
2016-04      20660
2016-05      20877
2016-06      20854
2016-07      21851
2016-08      22852
2016-09      26440
2016-10      25943
2016-11      29501
2016-12      26550
2017-01      26263
2017-02      26665
2017-03      69109
2017-04    1025

,msno,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date,is_cancel
0,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,22,395,1599,1599,0,2016-10-23,2018-02-06,0
1,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,41,30,99,99,1,2017-03-15,2017-04-15,0
2,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,39,30,149,149,1,2017-02-28,2017-04-19,0
3,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,39,30,149,149,1,2017-03-31,2017-05-19,0
4,+++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc=,41,30,149,149,1,2017-03-26,2017-04-26,0


,msno,city,bd,gender,registered_via,registration_init_time
0,Rb9UwLQTrxzBVwCB6+bCcSQWZ9JiNLC9dXtM1oEsZA8=,1,0,NaN,11,20110911
1,+tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=,1,0,NaN,7,20110914
2,cV358ssn7a0f7jZOwGNWS07wCKVqxyiImJUX6xcIwKw=,1,0,NaN,11,20110915
3,9bzDeJP6sQodK73K5CBlJ6fgIQzPeLnRl0p5B77XP+g=,1,0,NaN,11,20110915
4,WFLY3s7z4EZsieHCt63XrsdtfTEmJ+2PnnKLH5GY4Tk=,6,32,female,9,20110915


,msno,date,num_25,num_50,num_75,num_985,num_100,num_unq,total_secs
392000000,Z1Q9XLWoKo1S4m6P3y1hQ4uCLcj/TwHVn9a98SUvbbY=,20151206,3,5,2,1,26,12,7717.819
392000001,Z1Q9XLWoKo1S4m6P3y1hQ4uCLcj/TwHVn9a98SUvbbY=,20160206,0,0,0,0,1,1,295.419
392000002,Z1Q9XLWoKo1S4m6P3y1hQ4uCLcj/TwHVn9a98SUvbbY=,20160405,3,0,0,3,16,6,4591.957
392000003,Z1Q9XLWoKo1S4m6P3y1hQ4uCLcj/TwHVn9a98SUvbbY=,20160601,0,1,0,0,32,4,8226.944
392000004,Z1Q9XLWoKo1S4m6P3y1hQ4uCLcj/TwHVn9a98SUvbbY=,20160809,0,1,0,0,3,1,1027.429


,msno,date,num_25,num_50,num_75,num_985,num_100,num_unq,total_secs
18000000,3vvibcFgn9RgEE8qMyfIFX5C43j5BE9dIqDbOJnBXeg=,20170322,1,1,0,1,38,12,8186.414
18000001,dW5eUejrrbLeyp2SCDH6JbX8qwfYeVsPi3+Kt8x7gAk=,20170306,0,1,0,0,0,1,94.026
18000002,gLSXZDdgigYg1qnHqqpGNIaZkj/IeacAhScu534g7HI=,20170303,0,0,0,0,4,4,782.696
18000003,MLYOvvoMJAcSIT+xQq5y+RvfgHqvtUoeO5/+TntVJ30=,20170307,0,0,0,0,3,3,753.866
18000004,F41BuIqhu3JzMIzmRBFA8JW+/+QVSnUvzpTbcgWeAgk=,20170303,3,0,0,0,6,7,1597.488


In [18]:
# ===== 4. Standardize remaining date columns =====
# transactions_raw dates already parsed in cell 3.
# Here we handle members + logs.

def convert_to_datetime(series, date_format="%Y%m%d", name=""):
    """
    Robust YYYYMMDD to datetime conversion.
    Handles: integers, floats, strings.
    """
    s = series.astype("string").str.strip()
    s = s.str.replace(r"\.0$", "", regex=True)
    valid_mask = s.str.fullmatch(r"\d{8}", na=False)
    s = s.where(valid_mask, np.nan)
    result = pd.to_datetime(s, errors="coerce", format=date_format)
    nat_count = result.isna().sum()
    valid_count = (~result.isna()).sum()
    print(f"{name}: converted {valid_count} valid dates, {nat_count} invalid/missing")
    return result

# members registration time
if "registration_init_time" in members_v3.columns:
    members_v3["registration_init_time"] = convert_to_datetime(
        members_v3["registration_init_time"], name="registration_init_time"
    )

# user logs - historical
if "date" in user_logs_hist.columns:
    user_logs_hist["date"] = convert_to_datetime(
        user_logs_hist["date"], name="user_logs_hist[date]"
    )

# user logs - March
if "date" in user_logs_march.columns:
    user_logs_march["date"] = convert_to_datetime(
        user_logs_march["date"], name="user_logs_march[date]"
    )

print("\n" + "="*80)
print("DATE CONVERSION RESULTS")
print("="*80)

print("\ntransactions_raw (already parsed):")
print(f"  transaction_date  : {transactions_raw['transaction_date'].min()} to {transactions_raw['transaction_date'].max()}")
print(f"  expire_date       : {transactions_raw['membership_expire_date'].min()} to {transactions_raw['membership_expire_date'].max()}")

print("\nmembers_v3:")
if "registration_init_time" in members_v3.columns:
    print(f"  registration_time : {members_v3['registration_init_time'].min()} to {members_v3['registration_init_time'].max()}")

print("\nuser_logs_hist:")
print(f"  date : {user_logs_hist['date'].min()} to {user_logs_hist['date'].max()}")
print(f"  dtype: {user_logs_hist['date'].dtype}")

print("\nuser_logs_march:")
print(f"  date : {user_logs_march['date'].min()} to {user_logs_march['date'].max()}")
print(f"  dtype: {user_logs_march['date'].dtype}")


registration_init_time: converted 6769473 valid dates, 0 invalid/missing
user_logs_hist[date]: converted 106543 valid dates, 0 invalid/missing
user_logs_march[date]: converted 396362 valid dates, 0 invalid/missing

DATE CONVERSION RESULTS

transactions_raw (already parsed):
  transaction_date  : 2015-01-01 00:00:00 to 2017-03-31 00:00:00
  expire_date       : 1970-01-01 00:00:00 to 2036-10-15 00:00:00

members_v3:
  registration_time : 2004-03-26 00:00:00 to 2017-04-29 00:00:00

user_logs_hist:
  date : 2015-01-01 00:00:00 to 2017-02-28 00:00:00
  dtype: datetime64[ns]

user_logs_march:
  date : 2017-03-01 00:00:00 to 2017-03-31 00:00:00
  dtype: datetime64[ns]



## 5. Clean transactions core


In [19]:

# ===== 5. Clean transactions core =====
transactions_core = transactions_raw.copy()   # ← combined v1+v2, dates already parsed

essential_tx_cols = [c for c in ["msno", "transaction_date", "membership_expire_date"]
                     if c in transactions_core.columns]
transactions_core = transactions_core.dropna(subset=essential_tx_cols).copy()

for c in ["payment_method_id", "payment_plan_days", "plan_list_price",
           "actual_amount_paid", "is_auto_renew", "is_cancel"]:
    if c in transactions_core.columns:
        transactions_core[c] = pd.to_numeric(transactions_core[c], errors="coerce")

# business rule: cancellation rows with 0-day plan → zero amount
if all(c in transactions_core.columns for c in ["payment_plan_days", "is_cancel", "actual_amount_paid"]):
    mask = (transactions_core["payment_plan_days"] == 0) & (transactions_core["is_cancel"] == 1)
    transactions_core.loc[mask, "actual_amount_paid"] = 0

for c in ["payment_method_id", "payment_plan_days", "plan_list_price",
           "actual_amount_paid", "is_auto_renew", "is_cancel"]:
    if c in transactions_core.columns:
        transactions_core[c] = transactions_core[c].fillna(0)
    # cast binary flags to int8 — avoids float equality issues in FE label logic
for c in ["is_cancel", "is_auto_renew"]:
    if c in transactions_core.columns:
        transactions_core[c] = transactions_core[c].astype("int8")

transactions_core["expire_gap_days"] = (
    transactions_core["membership_expire_date"] - transactions_core["transaction_date"]
).dt.days
transactions_core["is_extreme_expiry"] = (
    transactions_core["expire_gap_days"] > 365
).astype("int8")

transactions_core = transactions_core.sort_values(
    [c for c in ["msno", "transaction_date", "membership_expire_date"]
     if c in transactions_core.columns]
).reset_index(drop=True)

print("transactions_core shape       :", transactions_core.shape)
print("transactions_core unique users:", transactions_core["msno"].nunique())
print("transaction_date range        :",
      transactions_core["transaction_date"].min(), "→",
      transactions_core["transaction_date"].max())
print("\nExpiry month distribution (2015-01 to 2017-06):")
print(
    transactions_core["membership_expire_date"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
    .loc["2015-01":"2017-06"]
)
display(transactions_core.head())
display(transactions_core["expire_gap_days"].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99, 0.999]
))


transactions_core shape       : (1978751, 11)
transactions_core unique users: 1315711
transaction_date range        : 2015-01-01 00:00:00 → 2017-03-31 00:00:00

Expiry month distribution (2015-01 to 2017-06):
membership_expire_date
2015-01        692
2015-02      10739
2015-03      13713
2015-04      15783
2015-05      14490
2015-06      18560
2015-07      16170
2015-08      17254
2015-09      18290
2015-10      19097
2015-11      20027
2015-12      22196
2016-01      22502
2016-02      20902
2016-03      23163
2016-04      20660
2016-05      20877
2016-06      20854
2016-07      21851
2016-08      22852
2016-09      26440
2016-10      25943
2016-11      29501
2016-12      26550
2017-01      26263
2017-02      26665
2017-03      69109
2017-04    1025980
2017-05     135530
2017-06      35990
Freq: M, Name: count, dtype: int64


,msno,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date,is_cancel,expire_gap_days,is_extreme_expiry
0,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,22,395,1599,1599,0,2016-10-23,2018-02-06,0,471,1
1,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,41,30,99,99,1,2017-03-15,2017-04-15,0,31,0
2,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,39,30,149,149,1,2017-02-28,2017-04-19,0,50,0
3,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,39,30,149,149,1,2017-03-31,2017-05-19,0,49,0
4,+++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc=,41,30,149,149,1,2017-03-26,2017-04-26,0,31,0


count    1.978751e+06
mean     9.558947e+01
std      2.103649e+02
min     -1.720300e+04
50%      3.100000e+01
90%      3.060000e+02
95%      4.170000e+02
99%      9.860000e+02
99.9%    1.663250e+03
max      7.303000e+03
Name: expire_gap_days, dtype: float64


## 6. Clean members core


In [20]:

# ===== 6. Clean members core =====
members_core = members_v3.copy()

if "bd" in members_core.columns:
    members_core["bd"] = members_core["bd"].where(members_core["bd"].between(10, 80), np.nan)
    members_core["bd_missing"] = members_core["bd"].isna().astype("int8")
    members_core["bd"] = members_core["bd"].fillna(members_core["bd"].median())

for c in ["city", "gender", "registered_via"]:
    if c in members_core.columns:
        members_core[f"{c}_missing"] = members_core[c].isna().astype("int8")
        members_core[c] = members_core[c].astype("string").fillna("Unknown")

if "registration_init_time" in members_core.columns:
    members_core["registration_year"]  = members_core["registration_init_time"].dt.year
    members_core["registration_month"] = members_core["registration_init_time"].dt.month
    members_core["registration_day"]   = members_core["registration_init_time"].dt.day
    members_core = members_core.drop(columns=["registration_init_time"])  # ← add this line

print("members_core shape:", members_core.shape)
print("members_core unique users:", members_core["msno"].nunique())
display(members_core.head())


members_core shape: (6769473, 12)
members_core unique users: 6769473


,msno,city,bd,gender,registered_via,bd_missing,city_missing,gender_missing,registered_via_missing,registration_year,registration_month,registration_day
0,Rb9UwLQTrxzBVwCB6+bCcSQWZ9JiNLC9dXtM1oEsZA8=,1,27.0,Unknown,11,1,0,1,0,2011,9,11
1,+tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=,1,27.0,Unknown,7,1,0,1,0,2011,9,14
2,cV358ssn7a0f7jZOwGNWS07wCKVqxyiImJUX6xcIwKw=,1,27.0,Unknown,11,1,0,1,0,2011,9,15
3,9bzDeJP6sQodK73K5CBlJ6fgIQzPeLnRl0p5B77XP+g=,1,27.0,Unknown,11,1,0,1,0,2011,9,15
4,WFLY3s7z4EZsieHCt63XrsdtfTEmJ+2PnnKLH5GY4Tk=,6,32.0,female,9,0,0,0,0,2011,9,15



## 7. Clean logs separately


In [21]:

# ===== 7. Clean logs separately =====
def clean_logs_table(df):
    out = df.copy()

    essential_cols = [c for c in ["msno", "date"] if c in out.columns]
    out = out.dropna(subset=essential_cols).copy()

    log_numeric_cols = ["num_25", "num_50", "num_75", "num_985", "num_100", "num_unq", "total_secs"]
    for c in log_numeric_cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0)

    play_cols = [c for c in ["num_25", "num_50", "num_75", "num_985", "num_100"] if c in out.columns]
    if len(play_cols) > 0:
        out["total_plays_proxy"] = out[play_cols].sum(axis=1)

    out = out.drop_duplicates().sort_values(
        [c for c in ["msno", "date"] if c in out.columns]
    ).reset_index(drop=True)
    return out

user_logs_hist_clean = clean_logs_table(user_logs_hist)
user_logs_march_clean = clean_logs_table(user_logs_march)

print("user_logs_hist_clean shape :", user_logs_hist_clean.shape)
print("user_logs_march_clean shape:", user_logs_march_clean.shape)

print("hist unique users :", user_logs_hist_clean["msno"].nunique())
print("march unique users:", user_logs_march_clean["msno"].nunique())

display(user_logs_hist_clean.head())
display(user_logs_march_clean.head())


user_logs_hist_clean shape : (106543, 10)
user_logs_march_clean shape: (396362, 10)
hist unique users : 22443
march unique users: 316345


,msno,date,num_25,num_50,num_75,num_985,num_100,num_unq,total_secs,total_plays_proxy
0,+++FOrTS7ab3tIgIh8eWwX4FqRv8w/FoiOuyXsFvphY=,2016-09-09,42,5,11,1,54,58,16826.994,113
1,++O9DLyAL6MB3wMaRqwAfZMC37SDo+uUExEoseVzc0U=,2016-06-27,11,1,1,1,11,22,3230.422,25
2,++O9DLyAL6MB3wMaRqwAfZMC37SDo+uUExEoseVzc0U=,2016-09-30,1,0,0,0,4,5,992.855,5
3,++O9DLyAL6MB3wMaRqwAfZMC37SDo+uUExEoseVzc0U=,2016-12-23,2,0,0,0,6,7,1444.316,8
4,++UGC5bVrIbJWXS5Q4B6Xxoj/yUsduvLdPSZx4tqcGk=,2015-11-05,9,1,0,0,44,48,12030.098,54


,msno,date,num_25,num_50,num_75,num_985,num_100,num_unq,total_secs,total_plays_proxy
0,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,2017-03-25,3,0,0,0,24,23,6182.491,27
1,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,2017-03-01,4,0,0,1,15,19,4140.721,20
2,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,2017-03-02,1,1,0,1,20,21,5340.569,23
3,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,2017-03-09,3,0,2,0,45,32,10049.509,50
4,++/9R3sX37CjxbY/AaGvbwr3QkwElKBCtSvVzhCBDOk=,2017-03-08,5,2,0,2,18,12,4512.357,27



## 8. Relationship checks between logs cohorts


In [22]:

# ===== 8. Relationship checks =====
hist_users = set(user_logs_hist_clean["msno"].astype("string").unique().tolist())
march_users = set(user_logs_march_clean["msno"].astype("string").unique().tolist())
overlap_users = hist_users & march_users

print("Historical logs users:", len(hist_users))
print("March logs users     :", len(march_users))
print("Overlap users        :", len(overlap_users))

if len(hist_users) > 0:
    print("Overlap ratio vs historical:", len(overlap_users) / len(hist_users))
if len(march_users) > 0:
    print("Overlap ratio vs March     :", len(overlap_users) / len(march_users))

print("\nHistorical logs month coverage:")
print(sorted(user_logs_hist_clean["date"].dt.to_period("M").astype(str).unique().tolist())[:12],
      "... total months =", user_logs_hist_clean["date"].dt.to_period("M").nunique())

print("\nMarch logs month coverage:")
print(sorted(user_logs_march_clean["date"].dt.to_period("M").astype(str).unique().tolist()),
      "... total months =", user_logs_march_clean["date"].dt.to_period("M").nunique())


Historical logs users: 22443
March logs users     : 316345
Overlap users        : 4081
Overlap ratio vs historical: 0.18183843514681639
Overlap ratio vs March     : 0.012900472585310341

Historical logs month coverage:
['2015-01', '2015-02', '2015-03', '2015-04', '2015-05', '2015-06', '2015-07', '2015-08', '2015-09', '2015-10', '2015-11', '2015-12'] ... total months = 26

March logs month coverage:
['2017-03'] ... total months = 1



## 9. Setup metadata for Feature Engineering


In [23]:

# ===== 9. Setup metadata for Feature Engineering =====
setup_metadata = {
    "population_rule": "users whose effective membership state expires in target month (Scala-aligned)",
    "train_months": "2015-01 to 2016-12",
    "train_months_n": 24,
    "val_months": "2017-01 to 2017-02",
    "val_months_n": 2,
    "inference_month": "2017-03",
    "inference_cutoff": "2017-03-31",
    "grace_days": 30,
    "label_rule": "no renewal within 30 days after effective expiry",
    "feature_rule": "features use data only up to month-end cutoff (no leakage)",
    "anti_leakage": True,
    "transactions_v1_source": "Data/transactions.parquet",
    "transactions_v2_source": "Data/transactions_v2.parquet",
    "transactions_combined": "Data/clean_transactions_core.parquet",
    "members_source": "Data/members_v3.csv",
    "logs_hist_source": "Data/user_logs.parquet",
    "logs_march_source": "Data/user_logs_v2.parquet",
    "core_model_uses_logs": False,
    "important_note": (
        "Core final model uses combined transactions (v1+v2) + members. "
        "Logs are kept as auxiliary branch only."
    ),
}
print(json.dumps(setup_metadata, indent=2))


{
  "population_rule": "users whose effective membership state expires in target month (Scala-aligned)",
  "train_months": "2015-01 to 2016-12",
  "train_months_n": 24,
  "val_months": "2017-01 to 2017-02",
  "val_months_n": 2,
  "inference_month": "2017-03",
  "inference_cutoff": "2017-03-31",
  "grace_days": 30,
  "label_rule": "no renewal within 30 days after effective expiry",
  "feature_rule": "features use data only up to month-end cutoff (no leakage)",
  "anti_leakage": true,
  "transactions_v1_source": "Data/transactions.parquet",
  "transactions_v2_source": "Data/transactions_v2.parquet",
  "transactions_combined": "Data/clean_transactions_core.parquet",
  "members_source": "Data/members_v3.csv",
  "logs_hist_source": "Data/user_logs.parquet",
  "logs_march_source": "Data/user_logs_v2.parquet",
  "core_model_uses_logs": false,
  "important_note": "Core final model uses combined transactions (v1+v2) + members. Logs are kept as auxiliary branch only."
}


In [24]:

# ===== 10. Save clean outputs =====
transactions_core.to_parquet(DATA_DIR / "clean_transactions_core.parquet", index=False)
members_core.to_parquet(DATA_DIR / "clean_members_core.parquet", index=False)
user_logs_hist_clean.to_parquet(DATA_DIR / "clean_user_logs_hist.parquet", index=False)
user_logs_march_clean.to_parquet(DATA_DIR / "clean_user_logs_march.parquet", index=False)

metadata_v3 = {
    "transactions_v1_rows"     : int(tx_v1.shape[0]),
    "transactions_v2_rows"     : int(tx_v2.shape[0]),
    "transactions_combined_rows": int(transactions_core.shape[0]),
    "members_rows"             : int(members_core.shape[0]),
    "user_logs_hist_rows"      : int(user_logs_hist_clean.shape[0]),
    "user_logs_march_rows"     : int(user_logs_march_clean.shape[0]),
    "transactions_users"       : int(transactions_core["msno"].nunique()),
    "members_users"            : int(members_core["msno"].nunique()),
    "user_logs_hist_users"     : int(user_logs_hist_clean["msno"].nunique()),
    "user_logs_march_users"    : int(user_logs_march_clean["msno"].nunique()),
    "columns_transactions_core": list(transactions_core.columns),
    "columns_members_core"     : list(members_core.columns),
    "columns_user_logs_hist"   : list(user_logs_hist_clean.columns),
    "columns_user_logs_march"  : list(user_logs_march_clean.columns),
    "setup_metadata"           : setup_metadata,
}

with open(DATA_DIR / "preprocessing_metadata_v3.json", "w", encoding="utf-8") as f:
    json.dump(metadata_v3, f, ensure_ascii=False, indent=2, default=str)

print("Saved:")
for name in [
    "clean_transactions_core.parquet",
    "clean_members_core.parquet",
    "clean_user_logs_hist.parquet",
    "clean_user_logs_march.parquet",
    "preprocessing_metadata_v3.json",
]:
    print(f"  {DATA_DIR / name}")


Saved:
  Data\clean_transactions_core.parquet
  Data\clean_members_core.parquet
  Data\clean_user_logs_hist.parquet
  Data\clean_user_logs_march.parquet
  Data\preprocessing_metadata_v3.json


In [25]:

# ===== 11. Final sanity =====
print("transactions_core users:", transactions_core["msno"].nunique())
print("members_core users     :", members_core["msno"].nunique())
print("user_logs_hist users   :", user_logs_hist_clean["msno"].nunique())
print("user_logs_march users  :", user_logs_march_clean["msno"].nunique())

print("\ntransactions_core date range:")
print(transactions_core["transaction_date"].min(), "→", transactions_core["transaction_date"].max())

print("\ntransactions_core expiry distribution (train+val window):")
print(
    transactions_core["membership_expire_date"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
    .loc["2015-01":"2017-03"]
)

print("\nuser_logs_hist date range:")
print(user_logs_hist_clean["date"].min(), "→", user_logs_hist_clean["date"].max())

print("\nuser_logs_march date range:")
print(user_logs_march_clean["date"].min(), "→", user_logs_march_clean["date"].max())

# ── v1 vs v2 overlap check ──
v1_users = set(tx_v1["msno"].astype(str).unique())
v2_users = set(tx_v2["msno"].astype(str).unique())
overlap  = v1_users & v2_users
print(f"\ntx_v1 users        : {len(v1_users):,}")
print(f"tx_v2 users        : {len(v2_users):,}")
print(f"overlap users      : {len(overlap):,}")
print(f"combined unique    : {len(v1_users | v2_users):,}")


transactions_core users: 1315711
members_core users     : 6769473
user_logs_hist users   : 22443
user_logs_march users  : 316345

transactions_core date range:
2015-01-01 00:00:00 → 2017-03-31 00:00:00

transactions_core expiry distribution (train+val window):
membership_expire_date
2015-01      692
2015-02    10739
2015-03    13713
2015-04    15783
2015-05    14490
2015-06    18560
2015-07    16170
2015-08    17254
2015-09    18290
2015-10    19097
2015-11    20027
2015-12    22196
2016-01    22502
2016-02    20902
2016-03    23163
2016-04    20660
2016-05    20877
2016-06    20854
2016-07    21851
2016-08    22852
2016-09    26440
2016-10    25943
2016-11    29501
2016-12    26550
2017-01    26263
2017-02    26665
2017-03    69109
Freq: M, Name: count, dtype: int64

user_logs_hist date range:
2015-01-01 00:00:00 → 2017-02-28 00:00:00

user_logs_march date range:
2017-03-01 00:00:00 → 2017-03-31 00:00:00

tx_v1 users        : 448,142
tx_v2 users        : 1,197,050
overlap users      :